In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import lance
from pathlib import Path
from psilia.rich_utils import get_console
import numpy as np
from psilia.data.utils import psi_glob, np_save_bytes, np_load_bytes, load_yaml
from psilia.data.lance import LanceTaker

console = get_console()

In [ ]:
path = Path('~/workspace/data/rosbags/')
lance_dirs = psi_glob(path/"**/lance", is_dir=True)
lance_path = lance_dirs["path"][0]
console.print(lance_dirs)

In [ ]:
from glob import glob
glob(str(lance_path / "*"))

In [ ]:
files = psi_glob(lance_path / "*")
console.print(files)

In [ ]:
from pandas import DataFrame

meta = load_yaml(lance_path / "metadata.yaml")
df = DataFrame(meta["datasets"].values())

console.print(df.sort_values("topic")[["topic", "schema", "path"]])

In [ ]:
import lance

# THIS IS ANNOYING, that's why I wrote the LanceTaker
ds = lance.dataset(lance_path / "zed-zed_node-right-image_rect_color.lance")
console.print(ds.take([0]).to_pydict())

In [ ]:
taker = LanceTaker(
    lance_path / "zed-zed_node-right-image_rect_color.lance"
)

# Easy convenient access
# By row ...
result = taker[:10]
ts, ims = result["t", "data"]
console.inspect(ts=ts, t0=ts[0], ims=ims, im0=ims[0])

# ...and by elapsed time in seconds. The result contains
# additional attributes and fields with the corresponding row index, and
# the time delta
result = taker.closest_elapsed(1.)

t, im, _dt, _i = result["t", "data", "__dt__", "__index__"]
console.inspect(
    t=t, 
    im=im, 
    _dt=_dt, 
    dt=result.time_delta, 
    _i=_i, 
    i=result.index
)

In [ ]:
taker[110:112]["log_time", "timestamp", "publish_time"]

In [ ]:
from psilia.data.lance import SuperTaker
from psilia.transforms import Transform
from psilia.vision import CameraIntrinsics

taker = SuperTaker({
        "left": lance_path / "zed-zed_node-left-image_rect_color.lance",
        "depth": lance_path / "zed-zed_node-depth-depth_registered.lance",
        "depth/confidence": lance_path / "zed-zed_node-depth-depth_registered.lance",
        "pose": lance_path / "zed-zed_node-pose.lance",
        "depth/camera_info": lance_path / "zed-zed_node-depth-camera_info.lance",
        "left/camera_info": lance_path / "zed-zed_node-left-camera_info.lance",
    },
    key_transforms = {
        "pose": [lambda d: {"tf": Transform.from_dict(d)}],
        "depth/camera_info": [lambda d: {"intr": CameraIntrinsics.from_camera_info_dict(d)}],
        "left/camera_info": [lambda d: {"intr": CameraIntrinsics.from_camera_info_dict(d)}],
    })


r = taker.closest_elapsed(1.0)
console.print(r["depth/camera_info/intr"])

In [ ]:
sorted(
    range(len(files)),
    key=lambda x: [f.split("-") for f in files["name"].to_list()][x]
)

In [ ]:
sorted(files["name"].to_list())

In [ ]:
import jax
import jax.numpy as jnp


tf = Transform(jnp.zeros(3), jnp.ones(4), child="a", parent="b")

jnp.save("./_tf.npy", tf.tree_flatten())

In [ ]:
jnp.load("./_tf.npy")